# CatBoost Raw-Feature Approach on Kaggle

Clones `approach/catboot`. Two algorithms, hard switch:

1. **Dictionary frequency-matching** -- if any word in train.txt (of the
   right length, consistent with the board + wrong guesses) still
   matches, guess the letter most common among those matches.
2. **26 separate CatBoost classifiers** (one per letter) -- used only
   when no dictionary word matches at all. Each one is trained on raw
   positional/pattern features (which position holds which letter or is
   still blank, which letters have been guessed, length, wrong-guess
   count) with no pre-computed candidate/ngram/neural signals involved --
   contrast with `approach/catboost-meta`, which deliberately reuses
   those signals instead of learning from raw features.

All words come from the competition's own train.txt; nothing substituted
or fabricated. No GPU needed -- CatBoost trains fine on CPU for this.

**Before running:** in the notebook's Settings panel (right sidebar), set
**Internet = On** (needed to `git clone`).

In [ ]:
import catboost
print('catboost', catboost.__version__)

In [ ]:
REPO_URL = "https://github.com/Sahoo-Achyutananda/MELTWATER---HACKATHON.git"
BRANCH = "approach/catboot"

!rm -rf repo
!git clone --branch $BRANCH --single-branch $REPO_URL repo
%cd repo/brand-buzzword-hackathon
!ls

## Use the official competition dataset

The cloned repo carries its own copy of train.txt/test.txt (downloaded from
this same competition earlier), but overwrite them here so this notebook
verifiably sources data straight from Kaggle's own `/kaggle/input/`, not an
external GitHub copy -- same content, no ambiguity for anyone reviewing it.

In [ ]:
import shutil
shutil.copy("/kaggle/input/competitions/brand-buzzword-hackathon/train.txt", "train.txt")
shutil.copy("/kaggle/input/competitions/brand-buzzword-hackathon/test.txt", "test.txt")
print("train.txt and test.txt overwritten with the official competition dataset from /kaggle/input/")

## Train the 26 per-letter classifiers

60,000 synthetic game states drawn from real train.txt words -- each
contributes one training row to every letter's classifier that hasn't
been guessed in that state. Trains and saves all 26 models to
`src/catboost_raw_models/`.

In [ ]:
!python src/train_catboost_raw.py --n-states 60000

## Validate

Same held-out-train.txt methodology as every other branch. Compare
against approach/candidate-ngram's plain dictionary+ngram fallback
(39-40%) -- this needs to beat that to show CatBoost's raw-feature
fallback is actually better than simple n-gram statistics for the out-
of-dictionary case.

In [ ]:
!python src/validate_catboost_raw.py --full

## Generate submission.csv

Plays the actual game against every word in test.txt.

In [ ]:
!python src/generate_submission_catboost_raw.py

## Save outputs

In [ ]:
import shutil
shutil.copytree("src/catboost_raw_models", "/kaggle/working/catboost_raw_models")
shutil.copy("submission.csv", "/kaggle/working/submission.csv")
print("saved catboost_raw_models/ and submission.csv to /kaggle/working/ -- download from the Output tab")